# Preparación y estructuración de datos – Top 50 Spotify Colombia 2024

**Evidencia:** GA1-220501094-01-AA1-EV07 – Preparación, limpieza y estructuración de datos del proyecto individual con Pandas
**Aprendiz:** Jeferson Estiven Vargas · **Instructor:** Carlos Enrique Navia Torres

**Pregunta analítica:** ¿Qué géneros musicales aparecen con más frecuencia en el Top 50 diario de Spotify Colombia durante 2024?

Este notebook parte del diagnóstico de `data_quality.ipynb` y aplica las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*:

1. **Lección 2:** limpieza y estandarización (pipeline con `pipe()` y validación con `assert`).
2. **Lección 3:** filtrado, ordenamiento, agrupación e indicadores.
3. **Lección 4:** combinación, estructuración y exportación a `data/processed/`.

> Los archivos de `data/raw/` **no se modifican**. Todo el resultado se guarda en `data/processed/`.

## 0. Configuración

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

# Rutas: el notebook está en notebooks/, por eso se sube un nivel con ../
RUTA_RAW = Path("../data/raw")
RUTA_PROCESSED = Path("../data/processed")
ARCHIVO_KAGGLE = RUTA_RAW / "universal_top_spotify_songs.csv"
ARCHIVO_CHARTS = RUTA_RAW / "regional-co-weekly-2024-01-11.csv"

# Alcance del proyecto
PAIS = "CO"
ANIO = 2024
TAMANO_LISTA = 50

# Resultados del diagnóstico (data_quality.ipynb) que la limpieza debe reproducir
FECHAS_MEZCLADAS_ESPERADAS = {"2024-03-14", "2024-03-16", "2024-03-18"}
DIAS_CONFIABLES_ESPERADOS = 346
FILAS_CHARTS_ESPERADAS = 200

print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.1.0


---
## 1. Lección 2 – Limpieza y estandarización de datos

Orden de trabajo del material: **duplicados → tipos → nulos → texto → duplicados implícitos → verificación**.
Primero se revisa cada problema con evidencia (secciones 1.2 a 1.6) y después se empaqueta la limpieza en funciones encadenadas con `pipe()` (1.7 a 1.10).

### 1.1 Carga de los datos crudos

In [2]:
kaggle = pd.read_csv(ARCHIVO_KAGGLE, low_memory=False)   # Dataset 1: Top 50 diario de 73 listas
charts = pd.read_csv(ARCHIVO_CHARTS)                     # Dataset 2: Top 200 semanal de Colombia

print("Dataset 1 – Kaggle:", kaggle.shape)
print("Dataset 2 – Charts:", charts.shape)

Dataset 1 – Kaggle: (2110316, 25)
Dataset 2 – Charts: (200, 9)


### 1.2 Estado inicial de los datos

La función `perfil_calidad()` resume la calidad de una tabla en una fila. Se usará otra vez al final para comparar el **antes y el después**.

In [3]:
CLAVE_LISTA = ["snapshot_date", "country", "daily_rank"]   # identifica una posición en una lista diaria


def contar_textos_irregulares(df):
    """Cuenta celdas de texto con espacio invisible (\xa0), espacios dobles o espacios en los extremos."""
    total = 0
    for columna in df.select_dtypes(include=["object", "str"]).columns:
        texto = df[columna].dropna().astype(str)
        irregular = (texto.str.contains("\xa0", regex=False)
                     | texto.str.contains("  ", regex=False)
                     | (texto != texto.str.strip()))
        total += int(irregular.sum())
    return total


def perfil_calidad(df, nombre):
    """Devuelve un diccionario con los indicadores de calidad de una tabla."""
    perfil = {
        "tabla": nombre,
        "filas": len(df),
        "columnas": df.shape[1],
        "celdas_nulas": int(df.isna().sum().sum()),
        "filas_duplicadas": int(df.duplicated().sum()),
        "textos_irregulares": contar_textos_irregulares(df),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
    }
    if set(CLAVE_LISTA).issubset(df.columns):
        perfil["posiciones_repetidas"] = int(df.duplicated(subset=CLAVE_LISTA).sum())
        perfil["duracion_cero"] = int((df["duration_ms"] == 0).sum())
    if "previous_rank" in df.columns:
        perfil["previous_rank_-1"] = int((df["previous_rank"] == -1).sum())
    return perfil


def tabla_faltantes(df):
    """Nulos y porcentaje por columna, solo para las columnas que tienen nulos."""
    t = pd.DataFrame({"nulos": df.isna().sum(),
                      "porcentaje (%)": (df.isna().mean() * 100).round(2)})
    return t[t["nulos"] > 0].sort_values("nulos", ascending=False)


# Subconjunto crudo de Colombia 2024: sirve para comparar con el resultado limpio en igualdad de condiciones
kaggle_co_crudo = kaggle[(kaggle["country"] == PAIS)
                         & kaggle["snapshot_date"].str.startswith(str(ANIO))]

estado_inicial = pd.DataFrame([
    perfil_calidad(kaggle, "Kaggle crudo (todas las listas)"),
    perfil_calidad(kaggle_co_crudo, f"Kaggle crudo ({PAIS} {ANIO})"),
    perfil_calidad(charts, "Charts crudo"),
])
estado_inicial

,tabla,filas,columnas,celdas_nulas,filas_duplicadas,textos_irregulares,memoria_mb,posiciones_repetidas,duracion_cero,previous_rank_-1
0,Kaggle crudo (todas las listas),2110316,25,30448,0,268,1224.1,3686.0,29.0,NaN
1,Kaggle crudo (CO 2024),17461,25,0,0,0,10.4,33.0,0.0,NaN
2,Charts crudo,200,9,0,0,0,0.1,NaN,NaN,44.0


**Interpretación:** el dataset de Kaggle tiene 2.110.316 filas, pero solo 17.461 (0,83 %) son de Colombia 2024. Los problemas de toda la tabla (30.448 nulos, 268 textos irregulares y 29 duraciones en 0) están en otras listas: en Colombia 2024 hay 0 nulos, 0 textos irregulares y 0 duraciones en 0. Los problemas que sí afectan al alcance son las 33 posiciones repetidas y las fechas guardadas como texto. En Charts, el único problema es el faltante disfrazado: 44 valores `-1` en `previous_rank`.

### 1.3 Duplicados

Se revisan dos tipos: filas **exactamente iguales** y **posiciones repetidas** (misma fecha, país y puesto con canciones distintas), el problema más grave que encontró el diagnóstico.

In [4]:
print("Kaggle – filas exactamente duplicadas:", kaggle.duplicated().sum())
print("Charts – filas exactamente duplicadas:", charts.duplicated().sum())
print("Charts – canciones (uri) repetidas:", charts["uri"].duplicated().sum())

# Posiciones repetidas en Colombia 2024: keep=False marca todas las copias para verlas juntas
repetidas = (kaggle_co_crudo[kaggle_co_crudo.duplicated(subset=CLAVE_LISTA, keep=False)]
             .sort_values(CLAVE_LISTA))
print("\nFilas de Colombia 2024 en posiciones repetidas:", len(repetidas))
print("Fechas afectadas:", sorted(repetidas["snapshot_date"].unique()))
print("Filas por cada fecha afectada:")
print(kaggle_co_crudo[kaggle_co_crudo["snapshot_date"].isin(repetidas["snapshot_date"])]
      .groupby("snapshot_date").size())
repetidas[["snapshot_date", "daily_rank", "name", "artists"]].head(6)

Kaggle – filas exactamente duplicadas: 0
Charts – filas exactamente duplicadas: 0
Charts – canciones (uri) repetidas: 0

Filas de Colombia 2024 en posiciones repetidas: 66
Fechas afectadas: ['2024-03-14', '2024-03-16', '2024-03-18']
Filas por cada fecha afectada:
snapshot_date
2024-03-14    54
2024-03-16    55
2024-03-18    52
dtype: int64


,snapshot_date,daily_rank,name,artists
1570022,2024-03-14,2,PERRO NEGRO,"Bad Bunny, Feid"
1570023,2024-03-14,2,SI SABE FERXXO,"Blessd, Feid"
1570028,2024-03-14,8,AMARGURA,KAROL G
1570029,2024-03-14,8,LA FALDA,Myke Towers
1570030,2024-03-14,10,Feliz Cumpleaños Ferxxo,Feid
1570031,2024-03-14,10,CLASSY 101,"Feid, Young Miko"


**Interpretación:** Colombia 2024 no tiene filas exactamente duplicadas, pero sí 33 posiciones repetidas (66 filas si se cuentan las dos copias) concentradas en 3 fechas: 14, 16 y 18 de marzo. Esas listas tienen 54, 55 y 52 filas en lugar de 50, con 33 canciones de más y 22 puestos vacíos. Los 346 días restantes suman exactamente 17.300 filas, 50 por día.

**Decisión:** excluir completas las fechas con posiciones repetidas. En esas listas no se sabe qué canción ocupó realmente cada puesto, y conservarlas contaría canciones de más. Las filas exactamente duplicadas, si aparecen, se eliminan con `drop_duplicates()`.

### 1.4 Tipos de datos

In [5]:
print(kaggle[["snapshot_date", "album_release_date", "daily_rank", "popularity", "duration_ms"]].dtypes)

# Prueba de conversión sin modificar la tabla: ¿cuántas fechas no se pueden convertir?
for columna in ["snapshot_date", "album_release_date"]:
    convertida = pd.to_datetime(kaggle_co_crudo[columna], format="ISO8601", errors="coerce")
    invalidas = convertida.isna().sum() - kaggle_co_crudo[columna].isna().sum()
    print(f"{columna}: valores no convertibles = {invalidas}")

print("\nMemoria de Kaggle crudo:", round(kaggle.memory_usage(deep=True).sum() / 1e6, 1), "MB")

snapshot_date           str
album_release_date      str
daily_rank            int64
popularity            int64
duration_ms           int64
dtype: object
snapshot_date: valores no convertibles = 0
album_release_date: valores no convertibles = 0

Memoria de Kaggle crudo: 1227.5 MB


**Interpretación:** las dos fechas están guardadas como texto (`str`), pero el 100 % de los valores de Colombia 2024 se pueden convertir, así que `pd.to_datetime()` no pierde información. Las columnas numéricas usan `int64` aunque sus valores caben en tipos más pequeños.

**Decisión:** convertir `snapshot_date` y `album_release_date` a fecha con `pd.to_datetime()`. Las columnas numéricas pequeñas (posición, movimientos, popularidad) pasan a enteros de menor tamaño (`int8`, `int16`) para ahorrar memoria, y las que deberán admitir nulos usan tipos *nullable* (`Int8`, `Int64`).

### 1.5 Valores faltantes: causa antes de decidir

In [6]:
print("Kaggle – todas las listas:")
display(tabla_faltantes(kaggle))

# Hipótesis del diagnóstico: los nulos de country son la lista Global, no un error
sin_pais = kaggle[kaggle["country"].isna()]
print("Filas sin país:", len(sin_pais), "| fechas distintas:", sin_pais["snapshot_date"].nunique(),
      "| filas por fecha:", sin_pais.groupby("snapshot_date").size().unique())

print(f"\nKaggle – solo {PAIS} {ANIO}:")
display(tabla_faltantes(kaggle_co_crudo))

print("Charts – nulos reales:", int(charts.isna().sum().sum()))
print("Charts – previous_rank = -1 (faltante disfrazado):", (charts["previous_rank"] == -1).sum())
print("Charts – de ellas, con weeks_on_chart == 1:",
      ((charts["previous_rank"] == -1) & (charts["weeks_on_chart"] == 1)).sum())

Kaggle – todas las listas:


,nulos,porcentaje (%)
country,28908,1.37
album_name,822,0.04
album_release_date,659,0.03
name,30,0.00
artists,29,0.00


Filas sin país: 28908 | fechas distintas: 578 | filas por fecha: [50 52 54 53 51 47]

Kaggle – solo CO 2024:


,nulos,porcentaje (%)


Charts – nulos reales: 0
Charts – previous_rank = -1 (faltante disfrazado): 44
Charts – de ellas, con weeks_on_chart == 1: 2


| Columna | Causa | Decisión |
| --- | --- | --- |
| `country` (Kaggle) | Las 28.908 filas sin país son la lista **Global**: 578 fechas con unas 50 filas cada una | Se eliminan al filtrar `country == "CO"`: no pertenecen al alcance |
| `name`, `artists`, `album_name` (Kaggle) | Canción sin datos de catálogo. **En Colombia 2024 hay 0**: los 30, 29 y 822 nulos están en otras listas | Regla preventiva: rellenar con `"Unknown"`, porque la canción sí estuvo en la lista y no se puede borrar |
| `album_release_date` (Kaggle) | Fecha no registrada. **En Colombia 2024 hay 0**: los 659 están en otras listas | Regla preventiva: conservar el nulo, porque inventar una fecha sesgaría cualquier análisis por año |
| Características de audio con `duration_ms == 0` | Valor imposible: la medición falló | Reemplazar por nulo (`NA`) solo en esas columnas; la fila se conserva |
| `previous_rank == -1` (Charts) | La canción no estaba en la lista la semana anterior: es nueva o vuelve a entrar (solo unas pocas tienen `weeks_on_chart == 1`) | Reemplazar por nulo (`NA`): `-1` no es una posición y distorsiona los promedios |

### 1.6 Textos y categorías

In [7]:
COLUMNAS_TEXTO_KAGGLE = ["name", "artists", "album_name"]
COLUMNAS_TEXTO_CHARTS = ["artist_names", "track_name", "source"]

print("Textos irregulares – Kaggle CO 2024:", contar_textos_irregulares(kaggle_co_crudo[COLUMNAS_TEXTO_KAGGLE]))
print("Textos irregulares – Charts:", contar_textos_irregulares(charts[COLUMNAS_TEXTO_CHARTS]))

print("\nFilas con varios artistas – Kaggle CO 2024:", kaggle_co_crudo["artists"].str.contains(", ", na=False).sum(),
      "de", len(kaggle_co_crudo))
print("Filas con varios artistas – Charts:", charts["artist_names"].str.contains(", ").sum(), "de", len(charts))

print("\nIdentificadores – Charts:", charts["uri"].head(2).tolist())
print("Identificadores – Kaggle:", kaggle_co_crudo["spotify_id"].head(2).tolist())

Textos irregulares – Kaggle CO 2024: 0
Textos irregulares – Charts: 0

Filas con varios artistas – Kaggle CO 2024: 9395 de 17461
Filas con varios artistas – Charts: 100 de 200

Identificadores – Charts: ['spotify:track:7bywjHOc0wSjGGbj04XbVi', 'spotify:track:7iQXYTyuG13aoeHxGG28Nh']
Identificadores – Kaggle: ['13BDiikG6y5o5cQTK0HpW6', '3mg9jYNC1NihbthlglzrWP']


**Decisiones:**
- Quitar espacios invisibles (`\xa0`), espacios dobles y espacios en los extremos.
- **No** cambiar mayúsculas con `.str.title()`: los nombres de artistas tienen una escritura oficial (`KAROL G`, `Feid`) que se necesita para unirlos con la tabla de géneros.
- Crear `main_artist` (el primer artista de la lista) y `artist_count`, conservando la columna original. Cada canción tendrá **un solo** género, el de su artista principal, y así no se cuenta dos veces.
- Extraer el `spotify_id` de `uri` en Charts para poder cruzar los dos datasets en la Lección 4.
- Las dos variantes de la disquera *Bichota* **se conservan y se documentan**: la disquera no interviene en la pregunta analítica.

### 1.7 Funciones de limpieza reutilizables

Cada función hace **una sola cosa**, recibe un DataFrame y devuelve otro. Esa regla permite encadenarlas con `pipe()`.

In [8]:
COLUMNAS_AUDIO = ["duration_ms", "danceability", "energy", "key", "loudness", "mode",
                  "speechiness", "acousticness", "instrumentalness", "liveness",
                  "valence", "tempo", "time_signature"]


def convertir_fechas(df):
    df = df.copy()
    df["snapshot_date"] = pd.to_datetime(df["snapshot_date"], format="%Y-%m-%d")
    df["album_release_date"] = pd.to_datetime(df["album_release_date"], format="ISO8601", errors="coerce")
    return df


def filtrar_pais_anio(df, pais, anio):
    mascara = (df["country"] == pais) & (df["snapshot_date"].dt.year == anio)
    return df[mascara].reset_index(drop=True)


def excluir_listas_mezcladas(df):
    """Quita las fechas en las que una misma posición tiene más de una canción."""
    repetidas = df.duplicated(subset=CLAVE_LISTA, keep=False)
    fechas = df.loc[repetidas, "snapshot_date"].unique()
    return df[~df["snapshot_date"].isin(fechas)].reset_index(drop=True)


def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)


def optimizar_tipos_kaggle(df):
    return df.astype({
        "daily_rank": "int8", "daily_movement": "int16", "weekly_movement": "int16",
        "popularity": "int8", "duration_ms": "Int64",
        "key": "Int8", "mode": "Int8", "time_signature": "Int8",
    })


def anular_audio_duracion_cero(df):
    """Las filas con duración 0 tienen las mediciones de audio vacías: se pasan a NA."""
    df = df.copy()
    duracion_cero = df["duration_ms"] == 0
    for columna in COLUMNAS_AUDIO:
        df[columna] = df[columna].mask(duracion_cero)
    return df


def estandarizar_textos(df, columnas):
    df = df.copy()
    for columna in columnas:
        df[columna] = (df[columna]
                       .str.replace("\xa0", " ", regex=False)
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip())
    return df


def rellenar_textos_nulos(df, columnas, valor="Unknown"):
    df = df.copy()
    df[columnas] = df[columnas].fillna(valor)
    return df


def agregar_artista_principal(df, columna_artistas):
    df = df.copy()
    artistas = df[columna_artistas].str.split(", ")
    df["main_artist"] = artistas.str[0]
    df["artist_count"] = artistas.str.len().astype("int8")
    return df


def previous_rank_a_nulo(df):
    df = df.copy()
    df["previous_rank"] = df["previous_rank"].astype("Int16").mask(df["previous_rank"] == -1)
    return df


def extraer_spotify_id(df):
    df = df.copy()
    df["spotify_id"] = df["uri"].str.split(":").str[-1]
    return df

### 1.8 Pipeline de limpieza con `pipe()` e informe de cambios

`anotar()` toma una "foto" de la tabla después de cada paso y la devuelve sin cambios. Por eso se puede intercalar entre los pasos.

In [9]:
registro = []   # se reinicia en la misma celda para que una segunda ejecución no duplique el informe


def anotar(df, tabla, paso):
    registro.append({
        "tabla": tabla,
        "paso": paso,
        "filas": len(df),
        "columnas": df.shape[1],
        "nulos": int(df.isna().sum().sum()),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df


kaggle_limpio = (kaggle
    .pipe(anotar, "kaggle", "0 crudo")
    .pipe(convertir_fechas)
    .pipe(filtrar_pais_anio, PAIS, ANIO)
    .pipe(anotar, "kaggle", f"1 fechas + filtro {PAIS} {ANIO}")
    .pipe(excluir_listas_mezcladas)
    .pipe(anotar, "kaggle", "2 sin listas mezcladas")
    .pipe(quitar_duplicados)
    .pipe(anotar, "kaggle", "3 sin duplicados exactos")
    .pipe(optimizar_tipos_kaggle)
    .pipe(anotar, "kaggle", "4 tipos optimizados")
    .pipe(anular_audio_duracion_cero)
    .pipe(anotar, "kaggle", "5 audio con duración 0 a NA")
    .pipe(estandarizar_textos, COLUMNAS_TEXTO_KAGGLE)
    .pipe(rellenar_textos_nulos, COLUMNAS_TEXTO_KAGGLE)
    .pipe(anotar, "kaggle", "6 textos estandarizados")
    .pipe(agregar_artista_principal, "artists")
    .pipe(anotar, "kaggle", "7 artista principal"))

charts_limpio = (charts
    .pipe(anotar, "charts", "0 crudo")
    .pipe(previous_rank_a_nulo)
    .pipe(anotar, "charts", "1 previous_rank -1 a NA")
    .pipe(estandarizar_textos, COLUMNAS_TEXTO_CHARTS)
    .pipe(extraer_spotify_id)
    .pipe(agregar_artista_principal, "artist_names")
    .pipe(anotar, "charts", "2 textos, spotify_id y artista principal"))

informe_cambios = pd.DataFrame(registro)
informe_cambios

,tabla,paso,filas,columnas,nulos,memoria_mb
0,kaggle,0 crudo,2110316,25,30448,1227.5
1,kaggle,1 fechas + filtro CO 2024,17461,25,0,8.5
2,kaggle,2 sin listas mezcladas,17300,25,0,8.4
3,kaggle,3 sin duplicados exactos,17300,25,0,8.4
4,kaggle,4 tipos optimizados,17300,25,0,7.7
5,kaggle,5 audio con duración 0 a NA,17300,25,0,7.7
6,kaggle,6 textos estandarizados,17300,25,0,7.5
7,kaggle,7 artista principal,17300,27,0,8.5
8,charts,0 crudo,200,9,0,0.1
9,charts,1 previous_rank -1 a NA,200,9,44,0.1


**Interpretación del informe de cambios:** el filtro de Colombia 2024 es el paso con más impacto: reduce la tabla de 2.110.316 a 17.461 filas, elimina los 30.448 nulos (que pertenecían a otras listas) y baja la memoria de unos 1.227 MB a 8,5 MB. Excluir las 3 fechas mezcladas quita 161 filas y deja 17.300 (346 días × 50 canciones). La optimización de tipos reduce la memoria de 8,4 a 7,7 MB. Quitar duplicados exactos, anular las duraciones en 0 y estandarizar textos no cambian los datos de Colombia 2024, porque esos problemas solo existen en otras listas: se conservan como reglas de protección para versiones futuras del dataset. Al final se agregan `main_artist` y `artist_count` (27 columnas). En Charts, los 44 valores `-1` pasan a ser nulos reales y se agregan `spotify_id`, `main_artist` y `artist_count`.

### 1.9 Duplicados implícitos

Algunos duplicados solo aparecen después de estandarizar: la misma canción con nombres escritos distinto, o el mismo nombre y artista con identificadores diferentes (reediciones o versiones).

In [10]:
canciones = kaggle_limpio.drop_duplicates("spotify_id")
nombres_por_id = kaggle_limpio.groupby("spotify_id")["name"].nunique()
ids_por_cancion = canciones.groupby(["name", "main_artist"])["spotify_id"].nunique()

print("Canciones únicas (spotify_id):", len(canciones))
print("spotify_id con más de un nombre:", (nombres_por_id > 1).sum())
print("Nombre + artista con más de un spotify_id:", (ids_por_cancion > 1).sum())
ids_por_cancion[ids_por_cancion > 1].head(10)

Canciones únicas (spotify_id): 428
spotify_id con más de un nombre: 2
Nombre + artista con más de un spotify_id: 38


name                 main_artist    
ADIVINO              Myke Towers        2
APT.                 ROSÉ               2
Adonay               Rodolfo Aicardi    2
Beautiful Things     Benson Boone       2
Bellakita            Conep              2
Boquita de Caramelo  Rodolfo Aicardi    2
Cantares de Navidad  Rodolfo Aicardi    2
Cariñito             Rodolfo Aicardi    2
Chismofilia          Luis Alfonso       2
Contentoso           Luis Alfonso       2
Name: spotify_id, dtype: int64

**Interpretación:** hay 428 `spotify_id` distintos. 2 `spotify_id` tienen dos nombres (títulos corregidos) y 38 canciones (nombre + artista principal) tienen dos `spotify_id`: la misma canción publicada como sencillo y en un álbum o compilado, como *APT.* de ROSÉ.

**Decisión:** las filas **se conservan**: cada una es una aparición real en el Top 50 y ambas versiones tienen el mismo artista principal, así que no cambian el género. Para contar **canciones distintas** se usará la combinación nombre + artista principal y no el `spotify_id`, para no contar dos veces la misma canción (390 canciones y no 428).

### 1.10 Validación con `assert`

Las reglas que deben cumplir los datos limpios se convierten en una prueba. Si una falla, la ejecución se detiene y muestra cuál regla no se cumplió.

In [11]:
def validar_kaggle(df):
    filas_por_dia = df.groupby("snapshot_date").size()
    assert set(df["country"].unique()) == {PAIS}, "hay filas de otro país"
    assert (df["snapshot_date"].dt.year == ANIO).all(), "hay fechas fuera del año"
    assert not df.duplicated(subset=CLAVE_LISTA).any(), "hay posiciones repetidas"
    assert df["daily_rank"].between(1, TAMANO_LISTA).all(), "posición fuera de rango"
    assert (filas_por_dia == TAMANO_LISTA).all(), f"hay días sin {TAMANO_LISTA} canciones"
    assert len(filas_por_dia) == DIAS_CONFIABLES_ESPERADOS, f"se esperaban {DIAS_CONFIABLES_ESPERADOS} días"
    assert not (df["duration_ms"] == 0).any(), "quedan duraciones en 0"
    assert df[COLUMNAS_TEXTO_KAGGLE].notna().all().all(), "quedan textos nulos"
    assert contar_textos_irregulares(df[COLUMNAS_TEXTO_KAGGLE]) == 0, "quedan espacios irregulares"
    assert df["main_artist"].notna().all() and (df["main_artist"] != "").all(), "canciones sin artista principal"
    return "Kaggle: validación superada"


def validar_charts(df):
    assert len(df) == FILAS_CHARTS_ESPERADAS, "cambió el número de filas"
    assert not (df["previous_rank"] == -1).any(), "quedan valores -1"
    assert df["spotify_id"].str.len().eq(22).all(), "spotify_id con formato inesperado"
    assert df["spotify_id"].is_unique, "canciones repetidas"
    assert contar_textos_irregulares(df[COLUMNAS_TEXTO_CHARTS]) == 0, "quedan espacios irregulares"
    return "Charts: validación superada"


fechas_excluidas = set(kaggle_co_crudo.loc[kaggle_co_crudo.duplicated(subset=CLAVE_LISTA, keep=False),
                                           "snapshot_date"])
assert fechas_excluidas == FECHAS_MEZCLADAS_ESPERADAS, f"fechas mezcladas inesperadas: {fechas_excluidas}"

print(validar_kaggle(kaggle_limpio))
print(validar_charts(charts_limpio))

Kaggle: validación superada
Charts: validación superada


### 1.11 Calidad antes y después de la limpieza

In [12]:
estado_final = pd.DataFrame([
    perfil_calidad(kaggle_limpio, f"Kaggle limpio ({PAIS} {ANIO})"),
    perfil_calidad(charts_limpio, "Charts limpio"),
])
comparacion_calidad = pd.concat([estado_inicial, estado_final], ignore_index=True)
comparacion_calidad

,tabla,filas,columnas,celdas_nulas,filas_duplicadas,textos_irregulares,memoria_mb,posiciones_repetidas,duracion_cero,previous_rank_-1
0,Kaggle crudo (todas las listas),2110316,25,30448,0,268,1224.1,3686.0,29.0,NaN
1,Kaggle crudo (CO 2024),17461,25,0,0,0,10.4,33.0,0.0,NaN
2,Charts crudo,200,9,0,0,0,0.1,NaN,NaN,44.0
3,Kaggle limpio (CO 2024),17300,27,0,0,0,8.7,0.0,0.0,NaN
4,Charts limpio,200,12,44,0,0,0.1,NaN,NaN,0.0


**Interpretación:** después de la limpieza, Kaggle Colombia 2024 queda con 17.300 filas (346 días × 50), 0 posiciones repetidas, 0 nulos, 0 textos irregulares y 0 duraciones en 0, con 2 columnas nuevas (`main_artist` y `artist_count`). Charts conserva sus 200 filas; sus 44 nulos son los antiguos `-1`, que ahora pandas reconoce como faltantes. Ambas tablas cumplen todas las reglas de validación de la sección 1.10.

**Resultado de la Lección 2:** `kaggle_limpio` y `charts_limpio` quedan validados y listos para el filtrado y la agrupación de la Lección 3.